In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Load cleaned transaction dataset
df = pd.read_csv("../datasets/cleaned_dataset.csv")

# Adapt dataset column names to the prediction pipeline
df = df.rename(columns={
    "Date": "date",
    "Amount": "amount"
})

# Clean data types
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df["amount"] = pd.to_numeric(df["amount"], errors="coerce")

# Remove invalid transactions
df = df.dropna(subset=["date", "amount"]).copy()

# Extract year and month from the actual transaction date
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month

# Aggregate transaction data month-wise
monthly = (
    df.groupby(["year", "month"])
      .agg(
          total_spending=("amount", "sum"),
          transaction_count=("amount", "count"),
          average_transaction=("amount", "mean")
      )
      .reset_index()
)

# Add seasonal features
monthly["month_sin"] = np.sin(
    2 * np.pi * monthly["month"] / 12
)

monthly["month_cos"] = np.cos(
    2 * np.pi * monthly["month"] / 12
)

# Sort chronologically
monthly = monthly.sort_values(
    ["year", "month"]
).reset_index(drop=True)

# Previous month's spending
monthly["previous_month_spending"] = (
    monthly["total_spending"].shift(1)
)

# Average spending of previous 3 months
monthly["rolling_3_month_avg"] = (
    monthly["total_spending"]
    .shift(1)
    .rolling(3)
    .mean()
)

# Remove rows without sufficient history
monthly_model = monthly.dropna().copy()

print("Total monthly records:", len(monthly))
print("Training records:", len(monthly_model))
print("\nFinal training dataset:")
display(monthly_model)

Total monthly records: 12
Training records: 9

Final training dataset:


,year,month,total_spending,transaction_count,average_transaction,month_sin,month_cos,previous_month_spending,rolling_3_month_avg
3,2025,4,7937485.11,622,12761.230080,8.660254e-01,-5.000000e-01,9390091.29,8.270530e+06
4,2025,5,8847821.63,683,12954.350849,5.000000e-01,-8.660254e-01,7937485.11,8.218469e+06
5,2025,6,9018844.54,689,13089.759855,1.224647e-16,-1.000000e+00,8847821.63,8.725133e+06
6,2025,7,8951073.24,694,12897.800058,-5.000000e-01,-8.660254e-01,9018844.54,8.601384e+06
7,2025,8,8692664.18,686,12671.522128,-8.660254e-01,-5.000000e-01,8951073.24,8.939246e+06
8,2025,9,8702808.39,661,13166.124644,-1.000000e+00,-1.836970e-16,8692664.18,8.887527e+06
9,2025,10,7022232.93,651,10786.840138,-8.660254e-01,5.000000e-01,8702808.39,8.782182e+06
10,2025,11,8039843.94,587,13696.497342,-5.000000e-01,8.660254e-01,7022232.93,8.139235e+06
11,2025,12,8391272.78,698,12021.880774,-2.449294e-16,1.000000e+00,8039843.94,7.921628e+06


In [2]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Features used by the prediction model
features = [
    "month",
    "transaction_count",
    "average_transaction",
    "month_sin",
    "month_cos",
    "previous_month_spending",
    "rolling_3_month_avg"
]

target = "total_spending"

X = monthly_model[features]
y = monthly_model[target]

# Time-based split
# Do NOT randomly shuffle because this is time-series data.
split_index = int(len(monthly_model) * 0.8)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

# Train Random Forest model
model = RandomForestRegressor(
    n_estimators=200,
    max_depth=8,
    random_state=42
)

model.fit(
    X_train,
    y_train
)

# Make predictions
predictions = model.predict(X_test)

# Evaluation
mae = mean_absolute_error(
    y_test,
    predictions
)

rmse = np.sqrt(
    mean_squared_error(
        y_test,
        predictions
    )
)

print("\nModel Evaluation")
print("----------------")
print("MAE :", round(mae, 2))
print("RMSE:", round(rmse, 2))

# Compare actual vs predicted
comparison = pd.DataFrame({
    "actual": y_test.values,
    "predicted": predictions
})

comparison

Training samples: 7
Testing samples: 2

Model Evaluation
----------------
MAE : 71444.34
RMSE: 96210.96


,actual,predicted
0,8039843.94,7.903962e+06
1,8391272.78,8.398279e+06


In [3]:
import joblib

MODEL_PATH = Path(
    "../models/monthly_expense_predictor.pkl"
)

MODEL_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)

joblib.dump(
    model,
    MODEL_PATH
)

print("Model saved successfully:")
print(MODEL_PATH)

Model saved successfully:
..\models\monthly_expense_predictor.pkl


In [4]:
print("Monthly spending statistics:")
print(monthly_model["total_spending"].describe())

print("\nMonthly spending values:")
print(
    monthly_model[
        ["year", "month", "total_spending"]
    ].to_string(index=False)
)

Monthly spending statistics:
count    9.000000e+00
mean     8.400450e+06
std      6.428927e+05
min      7.022233e+06
25%      8.039844e+06
50%      8.692664e+06
75%      8.847822e+06
max      9.018845e+06
Name: total_spending, dtype: float64

Monthly spending values:
 year  month  total_spending
 2025      4      7937485.11
 2025      5      8847821.63
 2025      6      9018844.54
 2025      7      8951073.24
 2025      8      8692664.18
 2025      9      8702808.39
 2025     10      7022232.93
 2025     11      8039843.94
 2025     12      8391272.78
